# 4.2 Demand Prediction

In this notebook we implement Neural Networks (NNs) to predict taxi trip demand in chicago. Additionally, we compare the performances of the NNs across different complexity levels.

For NNs there are 2 "main" complexity interpretations:
- depth: number of hidden layers
- width: number of nodes per layer
- (more complex activation function) - maybe as an extra
- (more complex optimizer) - maybe as an extra

So we decide to test 3 different NN structures:

__baseline model__:
- hidden layers: 2
- nodes per layer: 64

__wider model__:
- hidden layers: 2
- nodes per layer: 128

__deeper model__:
- hidden layers: 4
- nodes per layer: 64

For better comparison, we will test all three architectures with the same shared configurations.

In [41]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
# import matplotlib.pyplot as plt

# modeling
import copy
import random
import types

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

# reset working dir
import os
from pathlib import Path


In [42]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/anthony/Documents/Dokumente – MacBook Pro von Anthony/UNI/AAA/AAA_TA_2026


In [43]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data= pd.read_csv("data/aggregated/hexagon/demand_hex_1h_high.csv")

In [44]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Shared Configs                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# --- optimization ---
OPTIMIZER          = "adam"
LEARNING_RATE      = 1e-5
BATCH_SIZE         = 256
MAX_EPOCHS         = 100

# --- loss / output ---
LOSS               = "poisson_nll"     # demand is count data
OUTPUT_UNITS       = 1
OUTPUT_ACTIVATION  = "softplus"        # non-negative expected count

# --- layer defaults ---
HIDDEN_ACTIVATION  = "relu"
WEIGHT_INIT        = "he_normal"

# TODO: remove if not needed, depends on the degree of overfitting
# --- regularization (off by default to isolate the complexity effect) ---
DROPOUT            = 0.0
WEIGHT_DECAY       = 0.0

# --- early stopping ---
EARLY_STOPPING     = True
MONITOR            = "val_loss"
PATIENCE           = 10

# --- data handling ---
SPLIT              = "temporal"        # earlier 2025 -> train, later -> val/test
SCALER_FIT_ON      = "train_only"
SHUFFLE_TIME       = False             # never shuffle across time (no leakage)

# --- reproducibility ---
SEEDS              = (0, 1, 2, 3, 4)   # run each model across all seeds; report mean +/- std

# --- architectures (the ONLY thing that varies across the 3 models) ---
# format: (n_hidden_layers, width)
ARCH_BASELINE      = (2, 64)
ARCH_WIDER         = (2, 128)          # depth fixed, width up
ARCH_DEEPER        = (4, 64)           # width fixed, depth up
# expose as module-like object so model code can use config.XXX
config = types.SimpleNamespace(
    LEARNING_RATE  = LEARNING_RATE,
    WEIGHT_DECAY   = WEIGHT_DECAY,
    BATCH_SIZE     = BATCH_SIZE,
    MAX_EPOCHS     = MAX_EPOCHS,
    EARLY_STOPPING = EARLY_STOPPING,
    PATIENCE       = PATIENCE,
    OUTPUT_UNITS   = OUTPUT_UNITS,
)

ARCH_NAMES = {ARCH_BASELINE: "baseline", ARCH_WIDER: "wide", ARCH_DEEPER: "deep"}


In [45]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

class DemandBaseline(nn.Module):
    def __init__(self, input_dim, n_layers, width):
        super().__init__()
        layers = []
        in_dim = input_dim # tracks input size of next layer/number of outputs of current layer going into next layer
        for _ in range(n_layers):
            linear = nn.Linear(in_dim, width)
            nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")  # He initialization for ReLU, he/kaiming init keeps variance of weights constant
            nn.init.zeros_(linear.bias) # biases start at 0
            layers += [linear, nn.ReLU()] # hidden layers with ReLU activation
            in_dim = width
        out = nn.Linear(in_dim, config.OUTPUT_UNITS) # output layer maps to 1 unit
        nn.init.kaiming_normal_(out.weight, nonlinearity="relu")
        nn.init.zeros_(out.bias) # output layer bias starts at 0
        layers += [out, nn.Softplus()]   # forces the output to be non-negative
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1) # remove shape so only prediction is left


# --------------------------------------------------------------------------- #
# 2. Reproducibility
# --------------------------------------------------------------------------- #
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------------- #
# 3. Training loop (Adam + early stopping, all values from config)
# --------------------------------------------------------------------------- #
def train_model(arch, X_train, y_train, X_val, y_val, seed=0, device="cpu", verbose=True):
    set_seed(seed)
    n_layers, width = arch
    model = DemandBaseline(X_train.shape[1], n_layers, width).to(device)

    loss_fn = nn.PoissonNLLLoss(log_input=False, full=False)  # input is the rate
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY, # decay is 0 by default, no regularization
    )

    train_dl = DataLoader(
        TensorDataset(torch.as_tensor(X_train, dtype=torch.float32),
                      torch.as_tensor(y_train, dtype=torch.float32)),
        batch_size=config.BATCH_SIZE,
        shuffle=True,        # rows within the training set may shuffle; the
                             # train/val SPLIT itself stays temporal (no leakage)
    )
    X_val_t = torch.as_tensor(X_val, dtype=torch.float32).to(device)
    y_val_t = torch.as_tensor(y_val, dtype=torch.float32).to(device)

    best_val, best_state, wait = float("inf"), None, 0 # early stopping
    epoch_width = len(str(config.MAX_EPOCHS))

    for epoch in range(config.MAX_EPOCHS):
        # ── train ──────────────────────────────────────────────────────────────
        model.train()
        running_loss, n_batches = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches    += 1
        train_loss = running_loss / n_batches

        # ── validate ───────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()

        # ── early stopping bookkeeping ─────────────────────────────────────────
        improved = val_loss < best_val
        if improved:
            best_val, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1

        if verbose:
            marker = " *" if improved else f" (no improvement {wait}/{config.PATIENCE})"
            print(f"  epoch {epoch+1:{epoch_width}d}/{config.MAX_EPOCHS}"
                  f"  train={train_loss:.4f}  val={val_loss:.4f}{marker}")

        if config.EARLY_STOPPING and wait >= config.PATIENCE:
            if verbose:
                print(f"  early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)   # restore best weights
    return model, best_val


In [25]:
data.head()

,hour_stamp_since_epoch,pickup_h3_high_resolution,time_bucket,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,...,start_month_sin,start_month_cos,day_of_week_sin,day_of_week_cos,avg_trip_duration,avg_trip_distance,avg_fare,avg_trip_total,avg_tip,tip_rate
0,482136,882664d98bfffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
1,482136,882664c837fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
2,482136,8826645005fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
3,482136,882664d8dbfffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
4,482136,882664ce21fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
# Source - https://stackoverflow.com/a/18145399
# Posted by LondonRob, modified by community. See post 'Timeline' for change history
# Retrieved 2026-06-08, License - CC BY-SA 4.0

data = data.drop('hour_stamp_since_epoch', axis=1)


In [27]:
len(data)

17118994

In [28]:
data.columns

Index(['pickup_h3_high_resolution', 'time_bucket', 'trip_seconds',
       'trip_miles', 'fare', 'tips', 'tolls', 'extras', 'trip_total',
       'pickup_centroid_latitude', 'pickup_centroid_longitude',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'index',
       'relative_humidity_2m', 'surface_pressure', 'wind_speed_100m',
       'direct_radiation', 'avg_idle_time', 'rush_hour', 'trip_count',
       'active_taxis', 'start_month', 'start_hour', 'day_of_week',
       'is_weekend', 'is_rush_hour', 'is_holiday', 'temperature_2m',
       'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth',
       'wind_speed_10m', 'cloud_cover', 'is_day', 'rain', 'sunshine_duration',
       'start_hour_sin', 'start_hour_cos', 'start_month_sin',
       'start_month_cos', 'day_of_week_sin', 'day_of_week_cos',
       'avg_trip_duration', 'avg_trip_distance', 'avg_fare', 'avg_trip_total',
       'avg_tip', 'tip_rate'],
      dtype='str')

## Train / Val / Test Split

We split on **unique timestamps** (not rows) so that every hexagon for a given
hour lands in exactly one split — no leakage across the temporal boundary.

`TimeSeriesSplit(n_splits=2, test_size=≈15 %)` produces three contiguous segments:

| Split | Share | Period |
|-------|-------|--------|
| Train | ~70 % | Jan → early Oct |
| Val   | ~15 % | early Oct → mid Nov |
| Test  | ~15 % | mid Nov → Dec |

In [29]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Train / Val / Test Split                #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data['time_bucket'] = pd.to_datetime(data['time_bucket'], format='mixed')

timestamps = pd.DatetimeIndex(np.sort(data['time_bucket'].unique()))
n_ts = len(timestamps)

# ~70 / 15 / 15 split on unique timestamps
test_size = int(round(0.15 * n_ts))
tscv = TimeSeriesSplit(n_splits=2, test_size=test_size)
splits = list(tscv.split(timestamps))

# fold 0 → train | val,  fold 1 → (train+val) | test
train_ts = timestamps[splits[0][0]]
val_ts   = timestamps[splits[0][1]]
test_ts  = timestamps[splits[1][1]]

train_data = data[data['time_bucket'].isin(train_ts)].reset_index(drop=True)
val_data   = data[data['time_bucket'].isin(val_ts)].reset_index(drop=True)
test_data  = data[data['time_bucket'].isin(test_ts)].reset_index(drop=True)

print(f"Train : {len(train_ts):5d} timestamps  |  {train_ts[0].date()} → {train_ts[-1].date()}  |  {len(train_data):>9,} rows")
print(f"Val   : {len(val_ts):5d} timestamps  |  {val_ts[0].date()} → {val_ts[-1].date()}  |  {len(val_data):>9,} rows")
print(f"Test  : {len(test_ts):5d} timestamps  |  {test_ts[0].date()} → {test_ts[-1].date()}  |  {len(test_data):>9,} rows")


Train :  6133 timestamps  |  2025-01-01 → 2025-09-13  |  11,983,882 rows
Val   :  1314 timestamps  |  2025-09-13 → 2025-11-07  |  2,567,556 rows
Test  :  1314 timestamps  |  2025-11-07 → 2026-01-01  |  2,567,556 rows


## Feature Preparation

Drop leakage columns (aggregated trip statistics that are derived from the same
time-bucket), ID/index columns, and the target.  
Fit a `StandardScaler` on the **training set only** to avoid leakage into val/test.


In [30]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Feature Preparation                     #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# leakage: these are computed from the trips in the same bucket
LEAKAGE_COLS = [
    'hour_stamp_since_epoch', 'time_bucket', 'trip_seconds', 'trip_miles', 'fare', 'tips', 'tolls',
    'extras', 'trip_total', 'avg_idle_time', 'avg_trip_duration', 'avg_trip_distance',
    'avg_fare', 'avg_trip_total', 'avg_tip', 'tip_rate', 'active_taxis',
    'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
    'start_month', 'start_hour', 'day_of_week', 'is_rush_hour',
]

ID_COLS    = ['pickup_h3_high_resolution', 'time_bucket']
TARGET_COL = 'trip_count'

FEATURE_COLS = [
    c for c in train_data.columns
    if c not in LEAKAGE_COLS + ID_COLS + [TARGET_COL]
]
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_data[FEATURE_COLS].values)
X_val   = scaler.transform(val_data[FEATURE_COLS].values)
X_test  = scaler.transform(test_data[FEATURE_COLS].values)

y_train = train_data[TARGET_COL].values.astype(float)
y_val   = val_data[TARGET_COL].values.astype(float)
y_test  = test_data[TARGET_COL].values.astype(float)

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")


Features (26): ['pickup_centroid_latitude', 'pickup_centroid_longitude', 'index', 'relative_humidity_2m', 'surface_pressure', 'wind_speed_100m', 'direct_radiation', 'rush_hour', 'is_weekend', 'is_holiday', 'temperature_2m', 'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth', 'wind_speed_10m', 'cloud_cover', 'is_day', 'rain', 'sunshine_duration', 'start_hour_sin', 'start_hour_cos', 'start_month_sin', 'start_month_cos', 'day_of_week_sin', 'day_of_week_cos']
X_train : (11983882, 26)   y_train : (11983882,)
X_val   : (2567556, 26)     y_val   : (2567556,)
X_test  : (2567556, 26)    y_test  : (2567556,)


## Baseline Model — Training

Run `ARCH_BASELINE = (2 hidden layers, 64 units)` across all seeds and report
the mean ± std validation loss.


In [32]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Training               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}\n")

baseline_val_losses  = []
baseline_models      = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_BASELINE, X_train, y_train, X_val, y_val,
        seed=seed, device=device,
    )
    baseline_val_losses.append(val_loss)
    baseline_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nBaseline  val loss:  {np.mean(baseline_val_losses):.4f} ± {np.std(baseline_val_losses):.4f}")


Device: cpu

  epoch   1/100  train=-0.0915  val=-0.1545 *
  epoch   2/100  train=-0.1274  val=-0.1022 (no improvement 1/10)
  epoch   3/100  train=-0.1357  val=-0.1041 (no improvement 2/10)
  epoch   4/100  train=-0.1385  val=-0.1291 (no improvement 3/10)
  epoch   5/100  train=-0.1415  val=-0.0801 (no improvement 4/10)
  epoch   6/100  train=-0.1423  val=-0.0874 (no improvement 5/10)
  epoch   7/100  train=-0.1444  val=-0.0772 (no improvement 6/10)
  epoch   8/100  train=-0.1438  val=-0.0692 (no improvement 7/10)
  epoch   9/100  train=-0.1443  val=-0.0925 (no improvement 8/10)
  epoch  10/100  train=-0.1432  val=-0.0849 (no improvement 9/10)
  epoch  11/100  train=-0.1466  val=-0.0737 (no improvement 10/10)
  early stopping at epoch 11
  seed=0  val_loss=-0.1545
  epoch   1/100  train=-0.0897  val=-0.1413 *
  epoch   2/100  train=-0.1264  val=-0.1172 (no improvement 1/10)
  epoch   3/100  train=-0.1340  val=-0.1199 (no improvement 2/10)
  epoch   4/100  train=-0.1411  val=-0.1142 (n

## Baseline Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **MAPE** — mean absolute percentage error (relative, skip zeros)


In [34]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Evaluation             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t = torch.as_tensor(X_test, dtype=torch.float32).to(device)

run_results = []
mae_scores, rmse_scores, mape_scores = [], [], []
n_layers, width = ARCH_BASELINE

for seed, model in zip(SEEDS, baseline_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).cpu().numpy()

    mae  = np.mean(np.abs(preds - y_test))
    rmse = np.sqrt(np.mean((preds - y_test) ** 2))
    mask = y_test > 0
    mape = np.mean(np.abs((preds[mask] - y_test[mask]) / y_test[mask])) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)
    print(f"  seed={seed}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    run_results.append({
        "timestamp"     : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"         : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"      : n_layers,
        "width"         : width,
        "learning_rate" : LEARNING_RATE,
        "batch_size"    : BATCH_SIZE,
        "seed"          : seed,
        "val_loss"      : baseline_val_losses[seed],
        "mae"           : round(mae,  6),
        "rmse"          : round(rmse, 6),
        "mape"          : round(mape, 4),
    })

print(f"\nBaseline  MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"Baseline  RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"Baseline  MAPE : {np.mean(mape_scores):.2f}% ± {np.std(mape_scores):.2f}%")


  seed=0  MAE=0.5227  RMSE=3.9370  MAPE=148.92%
  seed=1  MAE=0.5650  RMSE=3.9455  MAPE=167.69%
  seed=2  MAE=0.5788  RMSE=3.9565  MAPE=166.86%
  seed=3  MAE=0.5451  RMSE=3.9390  MAPE=169.72%
  seed=4  MAE=0.6111  RMSE=3.9777  MAPE=217.33%

Baseline  MAE  : 0.5646 ± 0.0300
Baseline  RMSE : 3.9511 ± 0.0149
Baseline  MAPE : 174.10% ± 22.87%


seed=0  MAE=0.8582  RMSE=4.0996  MAPE=235.40%
  seed=1  MAE=0.9216  RMSE=4.1380  MAPE=288.11%
  seed=2  MAE=0.9802  RMSE=4.2733  MAPE=262.88%
  seed=3  MAE=1.1739  RMSE=4.4089  MAPE=361.21%
  seed=4  MAE=0.8553  RMSE=4.1133  MAPE=276.12%

Baseline  MAE  : 0.9578 ± 0.1174
Baseline  RMSE : 4.2066 ± 0.1185
Baseline  MAPE : 284.75% ± 42.06%

In [35]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Created data/results/nn_results.csv with 5 rows
          timestamp    model  n_layers  width  learning_rate  batch_size  seed  val_loss      mae     rmse     mape
2026-06-13T14:44:03 baseline         2     64          0.005         256     0 -0.154457 0.522698 3.936998 148.9188
2026-06-13T14:44:03 baseline         2     64          0.005         256     1 -0.141265 0.565017 3.945495 167.6884
2026-06-13T14:44:03 baseline         2     64          0.005         256     2 -0.141709 0.578841 3.956541 166.8561
2026-06-13T14:44:03 baseline         2     64          0.005         256     3 -0.156943 0.545149 3.939008 169.7217
2026-06-13T14:44:04 baseline         2     64          0.005         256     4 -0.147501 0.611058 3.977664 217.3335


## Deeper Model — Training

Run `ARCH_DEEPER = (4 hidden layers, 64 units)` across all seeds.
Width is fixed; depth doubles relative to the baseline.


In [38]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Deeper Model — Training                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

deeper_val_losses = []
deeper_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_DEEPER, X_train, y_train, X_val, y_val,
        seed=seed, device=device,
    )
    deeper_val_losses.append(val_loss)
    deeper_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nDeeper  val loss:  {np.mean(deeper_val_losses):.4f} ± {np.std(deeper_val_losses):.4f}")


  epoch   1/100  train=0.1588  val=0.1003 *
  epoch   2/100  train=0.0229  val=0.0258 *
  epoch   3/100  train=-0.0232  val=-0.0069 *
  epoch   4/100  train=-0.0505  val=-0.0311 *
  epoch   5/100  train=-0.0682  val=-0.0442 *
  epoch   6/100  train=-0.0819  val=-0.0555 *
  epoch   7/100  train=-0.0929  val=-0.0642 *
  epoch   8/100  train=-0.1017  val=-0.0668 *
  epoch   9/100  train=-0.1091  val=-0.0756 *
  epoch  10/100  train=-0.1153  val=-0.0821 *
  epoch  11/100  train=-0.1208  val=-0.0807 (no improvement 1/10)
  epoch  12/100  train=-0.1252  val=-0.0749 (no improvement 2/10)
  epoch  13/100  train=-0.1293  val=-0.0856 *
  epoch  14/100  train=-0.1330  val=-0.0824 (no improvement 1/10)
  epoch  15/100  train=-0.1363  val=-0.0653 (no improvement 2/10)
  epoch  16/100  train=-0.1393  val=-0.0865 *
  epoch  17/100  train=-0.1421  val=-0.0870 *
  epoch  18/100  train=-0.1446  val=-0.0773 (no improvement 1/10)
  epoch  19/100  train=-0.1469  val=-0.0809 (no improvement 2/10)
  epoch  2

## Deeper Model — Evaluation


In [39]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Deeper Model — Evaluation               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t = torch.as_tensor(X_test, dtype=torch.float32).to(device)

run_results = []
mae_scores, rmse_scores, mape_scores = [], [], []
n_layers, width = ARCH_DEEPER

for seed, model in zip(SEEDS, deeper_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).cpu().numpy()

    mae  = np.mean(np.abs(preds - y_test))
    rmse = np.sqrt(np.mean((preds - y_test) ** 2))
    mask = y_test > 0
    mape = np.mean(np.abs((preds[mask] - y_test[mask]) / y_test[mask])) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)
    print(f"  seed={seed}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    run_results.append({
        "timestamp"     : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"         : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"      : n_layers,
        "width"         : width,
        "learning_rate" : LEARNING_RATE,
        "batch_size"    : BATCH_SIZE,
        "seed"          : seed,
        "val_loss"      : deeper_val_losses[seed],
        "mae"           : round(mae,  6),
        "rmse"          : round(rmse, 6),
        "mape"          : round(mape, 4),
    })

print(f"\nDeeper  MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"Deeper  RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"Deeper  MAPE : {np.mean(mape_scores):.2f}% ± {np.std(mape_scores):.2f}%")


  seed=0  MAE=0.9337  RMSE=4.1772  MAPE=292.44%
  seed=1  MAE=1.0083  RMSE=4.2534  MAPE=335.97%
  seed=2  MAE=0.8816  RMSE=4.1837  MAPE=254.86%
  seed=3  MAE=1.0181  RMSE=4.2543  MAPE=307.52%
  seed=4  MAE=1.0278  RMSE=4.3720  MAPE=349.33%

Deeper  MAE  : 0.9739 ± 0.0569
Deeper  RMSE : 4.2481 ± 0.0701
Deeper  MAPE : 308.02% ± 33.34%


In [40]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Appended 5 rows to data/results/nn_results.csv
          timestamp model  n_layers  width  learning_rate  batch_size  seed  val_loss      mae     rmse     mape
2026-06-14T12:43:12  deep         4     64        0.00001         256     0 -0.087047 0.933722 4.177202 292.4367
2026-06-14T12:43:12  deep         4     64        0.00001         256     1 -0.056223 1.008265 4.253420 335.9738
2026-06-14T12:43:12  deep         4     64        0.00001         256     2 -0.071655 0.881631 4.183690 254.8624
2026-06-14T12:43:12  deep         4     64        0.00001         256     3 -0.038341 1.018100 4.254317 307.5181
2026-06-14T12:43:13  deep         4     64        0.00001         256     4 -0.079930 1.027786 4.371970 349.3288


## Wider Model — Training

Run `ARCH_WIDER = (2 hidden layers, 128 units)` across all seeds.
Depth is fixed; width doubles relative to the baseline.


In [46]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Wider Model — Training                  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

wider_val_losses = []
wider_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_WIDER, X_train, y_train, X_val, y_val,
        seed=seed, device=device,
    )
    wider_val_losses.append(val_loss)
    wider_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nWider  val loss:  {np.mean(wider_val_losses):.4f} ± {np.std(wider_val_losses):.4f}")


  epoch   1/100  train=0.1454  val=0.0737 *
  epoch   2/100  train=0.0401  val=0.0103 *
  epoch   3/100  train=0.0002  val=-0.0209 *
  epoch   4/100  train=-0.0251  val=-0.0412 *
  epoch   5/100  train=-0.0437  val=-0.0551 *
  epoch   6/100  train=-0.0584  val=-0.0660 *
  epoch   7/100  train=-0.0702  val=-0.0681 *
  epoch   8/100  train=-0.0801  val=-0.0772 *
  epoch   9/100  train=-0.0882  val=-0.0795 *
  epoch  10/100  train=-0.0949  val=-0.0845 *
  epoch  11/100  train=-0.1008  val=-0.0702 (no improvement 1/10)
  epoch  12/100  train=-0.1059  val=-0.0817 (no improvement 2/10)
  epoch  13/100  train=-0.1105  val=-0.0770 (no improvement 3/10)
  epoch  14/100  train=-0.1146  val=-0.0879 *
  epoch  15/100  train=-0.1182  val=-0.0775 (no improvement 1/10)
  epoch  16/100  train=-0.1216  val=-0.0795 (no improvement 2/10)
  epoch  17/100  train=-0.1246  val=-0.0815 (no improvement 3/10)
  epoch  18/100  train=-0.1274  val=-0.0692 (no improvement 4/10)
  epoch  19/100  train=-0.1301  val=-

## Wider Model — Evaluation


In [47]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Wider Model — Evaluation                #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t = torch.as_tensor(X_test, dtype=torch.float32).to(device)

run_results = []
mae_scores, rmse_scores, mape_scores = [], [], []
n_layers, width = ARCH_WIDER

for seed, model in zip(SEEDS, wider_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).cpu().numpy()

    mae  = np.mean(np.abs(preds - y_test))
    rmse = np.sqrt(np.mean((preds - y_test) ** 2))
    mask = y_test > 0
    mape = np.mean(np.abs((preds[mask] - y_test[mask]) / y_test[mask])) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)
    print(f"  seed={seed}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    run_results.append({
        "timestamp"     : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"         : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"      : n_layers,
        "width"         : width,
        "learning_rate" : LEARNING_RATE,
        "batch_size"    : BATCH_SIZE,
        "seed"          : seed,
        "val_loss"      : wider_val_losses[seed],
        "mae"           : round(mae,  6),
        "rmse"          : round(rmse, 6),
        "mape"          : round(mape, 4),
    })

print(f"\nWider  MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"Wider  RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"Wider  MAPE : {np.mean(mape_scores):.2f}% ± {np.std(mape_scores):.2f}%")


  seed=0  MAE=1.0248  RMSE=4.2213  MAPE=325.55%
  seed=1  MAE=0.9063  RMSE=4.1120  MAPE=250.50%
  seed=2  MAE=0.9073  RMSE=4.1121  MAPE=255.31%
  seed=3  MAE=1.1293  RMSE=4.3365  MAPE=339.84%
  seed=4  MAE=1.1424  RMSE=4.3310  MAPE=360.85%

Wider  MAE  : 1.0220 ± 0.1025
Wider  RMSE : 4.2226 ± 0.0992
Wider  MAPE : 306.41% ± 45.13%


In [48]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


Appended 5 rows to data/results/nn_results.csv
          timestamp model  n_layers  width  learning_rate  batch_size  seed  val_loss      mae     rmse     mape
2026-06-14T15:16:19  wide         2    128        0.00001         256     0 -0.087856 1.024753 4.221322 325.5545
2026-06-14T15:16:20  wide         2    128        0.00001         256     1 -0.068899 0.906256 4.112021 250.5003
2026-06-14T15:16:20  wide         2    128        0.00001         256     2 -0.066387 0.907339 4.112060 255.3093
2026-06-14T15:16:20  wide         2    128        0.00001         256     3 -0.088028 1.129276 4.336548 339.8401
2026-06-14T15:16:21  wide         2    128        0.00001         256     4 -0.078890 1.142359 4.330955 360.8458
